In [8]:
import pandas as pd
import sqlalchemy

engine = sqlalchemy.create_engine("sqlite+pysqlite:///data/orders_and_users.db")
inspector = sqlalchemy.inspect(engine)
print(inspector.get_table_names())
inspector.get_columns("orders")

['customers', 'orders']


[{'name': 'InvoiceNo',
  'type': VARCHAR(),
  'nullable': True,
  'default': None,
  'primary_key': 1},
 {'name': 'CustomerID',
  'type': VARCHAR(),
  'nullable': True,
  'default': None,
  'primary_key': 0},
 {'name': 'Description',
  'type': VARCHAR(),
  'nullable': True,
  'default': None,
  'primary_key': 0},
 {'name': 'Quantity',
  'type': INTEGER(),
  'nullable': True,
  'default': None,
  'primary_key': 0},
 {'name': 'UnitPrice',
  'type': FLOAT(),
  'nullable': True,
  'default': None,
  'primary_key': 0},
 {'name': 'Category',
  'type': VARCHAR(),
  'nullable': True,
  'default': None,
  'primary_key': 0},
 {'name': 'Discount',
  'type': FLOAT(),
  'nullable': True,
  'default': None,
  'primary_key': 0},
 {'name': 'PaymentMethod',
  'type': VARCHAR(),
  'nullable': True,
  'default': None,
  'primary_key': 0}]

In [2]:
pd.read_sql(sqlalchemy.text("SELECT DISTINCT Location FROM customers"), con=engine)

,Location
0,Kyiv
1,Brovary
2,Boryspil
3,Boyarka
4,Hostomel
5,Bucha
6,Irpin
7,Vyshneve


In [4]:
query = sqlalchemy.text("SELECT * FROM customers WHERE Location = :city")
kyiv_customers = pd.read_sql(query, con=engine, params={'city': 'Kyiv'})
kyiv_customers

,CustomerID,Genre,Age,Annual_Income,Spending_Score,Location
0,002c2626-e984-459b-b96e-f66b9bf1c8d4,Male,19,1500,39,Kyiv
1,01eac1fb-42ac-4ad4-afd6-fe01d3e2dcf5,Female,20,1600,6,Kyiv
2,037f2615-2d98-4d9d-ba88-1aa37263dc0e,Female,31,1700,40,Kyiv
3,04e44f80-9f4e-4fc9-a45c-a81097d07a02,Female,23,1800,94,Kyiv
4,07065bc8-5d9c-4177-82f9-f5e9b64444f1,Female,30,1900,72,Kyiv
5,07a005e9-45b1-446b-a374-c8490f7c5dfa,Male,67,1900,14,Kyiv
6,09c9ad28-738f-4e6c-b61b-6eda53404915,Male,22,2000,79,Kyiv
7,0ce0c94e-d220-46cd-833a-2f864ad1f56f,Male,20,2100,66,Kyiv
8,0e6164db-1030-47a5-bcc2-dfea819668af,Female,35,2300,98,Kyiv
9,0f6160e6-f9ce-457b-a231-46a0f7cd3a22,Male,25,2400,73,Kyiv


In [9]:
query = sqlalchemy.text("SELECT * FROM customers ORDER BY Spending_score DESC LIMIT 3")
top3_customers = pd.read_sql(query, con=engine)
top3_customers

,CustomerID,Genre,Age,Annual_Income,Spending_Score,Location
0,0818064c-782b-4b0e-9ceb-1517653713f9,Female,35,1900,99,Boryspil
1,0e6164db-1030-47a5-bcc2-dfea819668af,Female,35,2300,98,Kyiv
2,04e44f80-9f4e-4fc9-a45c-a81097d07a02,Female,23,1800,94,Kyiv


In [11]:
query = sqlalchemy.text("""
SELECT Description, SUM(Quantity) AS total_power_banks
FROM orders
WHERE Description = 'Power Bank'
GROUP BY Description
""")
power_bank_sold = pd.read_sql(query, con=engine)
power_bank_sold

,Description,total_power_banks
0,Power Bank,91


In [13]:
query = sqlalchemy.text("""
SELECT PaymentMethod, SUM(Quantity * UnitPrice * (1 - Discount)) AS total_sum
FROM orders
WHERE PaymentMethod = 'credit card'
GROUP BY PaymentMethod
""")
credit_card_total = pd.read_sql(query, con=engine)
credit_card_total

,PaymentMethod,total_sum
0,credit card,8187.4637
